In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Dataseti oxuyuruq
file_path = Path("../01_Raw_Data/marketing_data.xlsx")

df_raw = pd.read_excel(file_path)

# İşləmək üçün surət yaradırıq
df = df_raw.copy()

# Sütun adlarını standartlaşdırırıq
df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (2240, 28)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,1826,1970,Graduation,Divorced,84835.0,0,0,2014-06-16,0,189,...,6,1,0,0,0,0,0,1,0,Spain
1,1,1961,Graduation,Single,57091.0,0,0,2014-06-15,0,464,...,7,5,0,0,0,0,1,1,0,Canada
2,10476,1958,Graduation,Married,67267.0,0,1,2014-05-13,0,134,...,5,2,0,0,0,0,0,0,0,USA
3,1386,1967,Graduation,Together,32474.0,1,1,2014-05-11,0,10,...,2,7,0,0,0,0,0,0,0,Australia
4,5371,1989,Graduation,Single,21474.0,1,0,2014-04-08,0,6,...,2,7,1,0,0,0,0,1,0,Spain


In [2]:
categorical_columns = [
    "Education",
    "Marital_Status",
    "Country"
]

for col in categorical_columns:

    print(f"\n===== {col} =====")

    print("Unique Values:", df[col].nunique())

    display(
        df[col].value_counts(dropna=False)
    )


===== Education =====
Unique Values: 5


Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64


===== Marital_Status =====
Unique Values: 8


Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
YOLO          2
Absurd        2
Name: count, dtype: int64


===== Country =====
Unique Values: 8


Country
Spain           1095
Saudi Arabia     337
Canada           268
Australia        160
India            148
Germany          120
USA              109
Mexico             3
Name: count, dtype: int64

In [3]:
for col in categorical_columns:

    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
    )

In [4]:
for col in categorical_columns:

    df[col] = (
        df[col]
        .str.replace(r"\s+", " ", regex=True)
    )

In [5]:
display(
    df["Education"].value_counts(dropna=False)
)

Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: Int64

In [6]:
education_mapping = {
    "Basic": "Basic Education",
    "2n Cycle": "Second Cycle",
    "Graduation": "Graduate",
    "Master": "Master's Degree",
    "PhD": "Doctorate"
}

df["Education_Standardized"] = (
    df["Education"].replace(education_mapping)
)

display(
    df[
        ["Education", "Education_Standardized"]
    ].drop_duplicates()
)

,Education,Education_Standardized
0,Graduation,Graduate
5,PhD,Doctorate
6,2n Cycle,Second Cycle
11,Master,Master's Degree
54,Basic,Basic Education


In [7]:
display(
    df["Marital_Status"].value_counts(dropna=False)
)

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
YOLO          2
Absurd        2
Name: count, dtype: Int64

In [8]:
marital_mapping = {
    "Married": "Married",
    "Together": "Partnered",
    "Single": "Single",
    "Alone": "Living Alone",
    "Divorced": "Divorced",
    "Widow": "Widowed",
    "Absurd": "Unknown",
    "YOLO": "Unknown"
}

df["Marital_Status_Group"] = (
    df["Marital_Status"].map(marital_mapping)
)

display(
    df[
        ["Marital_Status", "Marital_Status_Group"]
    ].drop_duplicates()
)

,Marital_Status,Marital_Status_Group
0,Divorced,Divorced
1,Single,Single
2,Married,Married
3,Together,Partnered
12,Widow,Widowed
103,YOLO,Unknown
286,Alone,Living Alone
1068,Absurd,Unknown


In [9]:
display(
    df["Country"].value_counts(dropna=False)
)

Country
Spain           1095
Saudi Arabia     337
Canada           268
Australia        160
India            148
Germany          120
USA              109
Mexico             3
Name: count, dtype: Int64

In [10]:
df["Country_Standardized"] = (
    df["Country"].str.title()
)

In [11]:
print(
    "Unmapped Education:",
    df["Education_Standardized"].isna().sum()
)

print(
    "Unmapped Marital Status:",
    df["Marital_Status_Group"].isna().sum()
)

print(
    "Missing Country:",
    df["Country_Standardized"].isna().sum()
)

Unmapped Education: 0
Unmapped Marital Status: 0
Missing Country: 0


In [12]:
display(
    df[
        df["Marital_Status_Group"].isna()
    ][
        ["ID", "Marital_Status"]
    ]
)

,ID,Marital_Status


In [13]:
category_comparison = pd.DataFrame({
    "Column": [
        "Education",
        "Marital_Status",
        "Country"
    ],
    "Original_Unique": [
        df_raw["Education"].nunique(),
        df_raw["Marital_Status"].nunique(),
        df_raw["Country"].nunique()
    ],
    "Standardized_Unique": [
        df["Education_Standardized"].nunique(),
        df["Marital_Status_Group"].nunique(),
        df["Country_Standardized"].nunique()
    ]
})

display(category_comparison)

,Column,Original_Unique,Standardized_Unique
0,Education,5,5
1,Marital_Status,8,7
2,Country,8,8


In [14]:
categorical_log = pd.DataFrame([
    {
        "Step": 7,
        "Column": "Education",
        "Issue": "Non-standard category labels",
        "Action": "Create standardized education labels",
        "Method": "replace()",
        "Status": "Completed"
    },
    {
        "Step": 8,
        "Column": "Marital_Status",
        "Issue": "Ambiguous category values",
        "Action": "Create analytical category groups",
        "Method": "map()",
        "Status": "Completed"
    },
    {
        "Step": 9,
        "Column": "Country",
        "Issue": "Potential inconsistent capitalization",
        "Action": "Standardize capitalization",
        "Method": "str.title()",
        "Status": "Completed"
    }
])

display(categorical_log)

,Step,Column,Issue,Action,Method,Status
0,7,Education,Non-standard category labels,Create standardized education labels,replace(),Completed
1,8,Marital_Status,Ambiguous category values,Create analytical category groups,map(),Completed
2,9,Country,Potential inconsistent capitalization,Standardize capitalization,str.title(),Completed
